In [2]:
import json
import os
from itertools import islice
from pathlib import Path
from pickle import dump, load
from time import time

import numpy as np
import pyspark
import pyspark.sql.functions as sf
from colabfit.tools.property_definitions import (
    atomic_forces_pd,
    cauchy_stress_pd,
    energy_pd,
)
from colabfit.tools.vast.configuration import AtomicConfiguration
from colabfit.tools.vast.configuration_set import ConfigurationSet
from colabfit.tools.vast.database import DataManager, VastDataLoader
from colabfit.tools.vast.property import PropertyMap, property_info
from colabfit.tools.vast.utilities import unstring_df_val
from dotenv import load_dotenv
from pymatgen.core import Structure
from pyspark.sql import Row, SparkSession
from pyspark.sql.types import (
    ArrayType,
    FloatType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)
from tqdm import tqdm

In [ ]:
from colabfit.tools.vast.database import generate_ds_id

# generate_ds_id()
# DS_xnio123pebli_0 for alexandria 1d

Generated new DS ID: DS_xnio123pebli_0


'DS_xnio123pebli_0'

In [ ]:
@sf.udf("string")
def make_desc(cs_name):
    ds_name, filename, mat_id, traj = cs_name.split("__")
    filename = filename.replace("file_", "")
    mat_id = mat_id.replace("id_", "")
    traj = traj.replace("trajectory_", "")
    desc = f"Alexandria 3D PBE trajectory number {traj} for material {mat_id} from file {filename}"
    return desc

### Alexandria 1D

In [5]:
with open(
    "alexandria_1d_pickles/alexandria_1d/alexandria_1d_alex_go_aaa_000_0.pkl", "rb"
) as f:
    configs = load(f)

In [ ]:
def read_alexandria(fp: Path):
    with fp.open("rb") as f:
        for config in load(f):
            yield config


DATASET_NAME = "Alexandria_geometry_optimization_paths_PBE_1D"


def read_wrapper(dir_path: str):
    dir_path = Path(dir_path)
    if not dir_path.exists():
        return
    all_json_paths = sorted(list(dir_path.rglob("*.pkl")), key=lambda x: x.stem)

    path = all_json_paths[0]
    css_dir = Path("alexandria_1d_css")
    css_dir.mkdir(exist_ok=True)
    css_path = css_dir / f"{path.stem}.csv"
    with css_path.open("a") as f:
        f.write("cs_name,config_id\n")
        reader = read_alexandria(path)
        for config in reader:
            _, filename, mat_id, traj, ix = config.info["_name"].split("__")
            cs_name = f"{DATASET_NAME}__{filename}__{mat_id}__{traj}"
            f.write(f"{cs_name},{config.id}\n")
            yield config

In [28]:
configs = read_wrapper("alexandria_1d_pickles/alexandria_1d/")

In [29]:
clist = list(configs)

### Alexandria 3D

In [ ]:
cs_names = css.withColumn("description", make_desc(sf.col("cs_name")))
dataset_id = "DS_s6gf4z2hcjqy_0"
cos = spark.table("ndb.colabfit.dev.co_wip")
config_df = cos.join(css.distinct(), cos.id == css.config_id, how="inner").drop(
    "config_id"
)
string_cols = [
    "elements",
]
unstring_col_udf = sf.udf(unstring_df_val, ArrayType(StringType()))
row_dict["last_modified"] = dateutil.parser.parse(
    datetime.now().strftime("%Y-%m-%d %H:%M:%S")
)
for col in string_cols:
    config_df = config_df.withColumn(col, unstring_col_udf(sf.col(col)))

unstring_col_udf = sf.udf(unstring_df_val, ArrayType(IntegerType()))
int_cols = [
    "atomic_numbers",
    "dimension_types",
]

for col in int_cols:
    config_df = config_df.withColumn(col, unstring_col_udf(sf.col(col)))

config_df.cache()
grouped = config_df.groupBy("cs_name")

result = grouped.agg(
    sf.sum(sf.col("nsites")).alias("nsites"),
    sf.count(sf.col("id")).alias("nconfigurations"),
    sf.collect_set("nperiodic_dimensions").alias("nperiodic_dimensions"),
    sf.collect_set("dimension_types").alias("dimension_types"),
    sf.sort_array(
        sf.collect_set(sf.explode(sf.col("elements"))).alias("exploded_elements")
    ),
)
# cs_names = cs_names.withColumn()
# cs = ConfigurationSet()
"""
Original files: 370
11000 files each with 200-400 cs names (about 10000 configurations)
divided into directories based on original file name
cs1,co1
cs1,co2
cs2,co1
group by cs-id
write out to files
spark map over cs
sdk to get cos
aggregate over pyarrow or pandas

"""

In [ ]:
for n in (
    cs_names.select("cs_name", "description")
    .distinct()
    .withColumn("description", make_desc(sf.col("cs_name")))
    .collect()
):
    name = n["cs_name"]
    desc = n["description"]
    configs = config_df.filter(sf.col("cs_name") == name)
    cs = ConfigurationSet(config_df=configs, name=name, description=desc)
    cs["ordered"] = True
    break

In [60]:
with open("alex_go_aad_020.json") as f:
    data = json.load(f)

In [44]:
with open("alexandria_3d_alex_go_aad_020_47.pkl", "rb") as f:
    configs = load(f)

In [53]:
len(configs)

10000

In [55]:
for i, c in enumerate(configs):
    try:
        x = np.atleast_1d(c.info["forces"])
    except:
        print(i)
        print(c.info)
        print(c.info["forces"])

85
{'_name': ['alexandria_3d__file_alex_go_aad_020__id_agm005676148__trajectory_0__frame_5'], 'energy': -105.92396102, 'forces': [[-0.00021793, -0.00044876, -0.00082885], [0.00186276, 0.00383572, 0.00708449], [0.00021793, 0.00044876, 0.00082885], [-0.00186276, -0.00383572, -0.00708449], [-0.00286273, -0.00589483, -0.01088763], [0.00286273, 0.00589483, 0.01088763], [0.00265601, 0.00546915, 0.01010142], [-0.00265601, -0.00546915, -0.01010142], [0.00943903], [-0.00365626, -0.00466598, 0.00767191], [-0.00527149, -0.00799199, 0.00152884], [0.00365626, 0.00466598, -0.00767191], [0.00131429, -0.00015313, 0.0004018], [0.00105779, -0.0006813, -0.00057372], [-0.00131429, 0.00015313, -0.0004018], [-0.00105779, 0.0006813, 0.00057372]], 'stress': [[-0.01798887, 0.00533262, 0.0280218], [0.00533257, 0.05793227, 0.02113897], [0.02802183, 0.02113896, 0.0807522]], 'input': {'kpoints': [4, 4, 3], 'ENAUG': 342.924, 'ENMAX': 337.5, 'PREC': 'high'}}
[[-0.00021793, -0.00044876, -0.00082885], [0.00186276, 0.0

In [56]:
x = configs[85]

In [57]:
x.info

{'_name': ['alexandria_3d__file_alex_go_aad_020__id_agm005676148__trajectory_0__frame_5'],
 'energy': -105.92396102,
 'forces': [[-0.00021793, -0.00044876, -0.00082885],
  [0.00186276, 0.00383572, 0.00708449],
  [0.00021793, 0.00044876, 0.00082885],
  [-0.00186276, -0.00383572, -0.00708449],
  [-0.00286273, -0.00589483, -0.01088763],
  [0.00286273, 0.00589483, 0.01088763],
  [0.00265601, 0.00546915, 0.01010142],
  [-0.00265601, -0.00546915, -0.01010142],
  [0.00943903],
  [-0.00365626, -0.00466598, 0.00767191],
  [-0.00527149, -0.00799199, 0.00152884],
  [0.00365626, 0.00466598, -0.00767191],
  [0.00131429, -0.00015313, 0.0004018],
  [0.00105779, -0.0006813, -0.00057372],
  [-0.00131429, 0.00015313, -0.0004018],
  [-0.00105779, 0.0006813, 0.00057372]],
 'stress': [[-0.01798887, 0.00533262, 0.0280218],
  [0.00533257, 0.05793227, 0.02113897],
  [0.02802183, 0.02113896, 0.0807522]],
 'input': {'kpoints': [4, 4, 3],
  'ENAUG': 342.924,
  'ENMAX': 337.5,
  'PREC': 'high'}}

In [2]:
CSS = dict()
DATASET_NAME = "Alexandria_geometry_optimization_paths_PBE_3D"


def read_alexandria(fp: Path):
    with fp.open("r") as f:
        data = json.load(f)
        print(data.keys())
    for ix, id in enumerate(data):
        val = data[id]
        for j, trajectory in enumerate(val):
            # cs_name_match = f"alexandria_3d__file_{fp.stem}__id_{id}__trajectory_{j}"
            # cs_label_match = None
            cs_name = f"{DATASET_NAME}__{fp.stem}__{id}__{j}"
            cs_description = f"Alexandria 3D PBE trajectory number {j} for material {id} from file {fp.name}"  # noqa
            CSS[cs_name] = {}
            CSS[cs_name]["name"] = cs_name
            CSS[cs_name]["description"] = cs_description
            CSS[cs_name]["ids"] = []
            # (cs_name_match, cs_label_match, cs_name, cs_description)]
            input = {
                "kpoints": trajectory["kpoints"],
                "ENAUG": trajectory["ENAUG"],
                "ENMAX": trajectory["ENMAX"],
                "PREC": trajectory["PREC"],
            }
            for i, step in enumerate(trajectory["steps"]):
                struct = Structure.from_dict(step["structure"])
                config = struct.to_ase_atoms()
                labels = [
                    f"alexandria_id:{id}",
                ]
                config.info["_name"] = (
                    f"alexandria_3d__file_{fp.stem}__id_{id}__trajectory_{j}__frame_{i}"
                )
                config.info["_labels"] = labels
                config.info["energy"] = step["energy"]
                config.info["forces"] = step["forces"]
                config.info["stress"] = step["stress"]
                config.info["input"] = input
                aconfig = AtomicConfiguration.from_ase(config)
                CSS[cs_name]["ids"].append(aconfig.id)
                yield aconfig


def batched(configs, n):
    "Batch data into tuples of length n. The last batch may be shorter."
    configs = iter(configs)
    while True:
        batch = list(islice(configs, n))
        if len(batch) == 0:
            break
        yield batch


def to_pickles(fp: Path):
    limit = 10000
    gener = read_alexandria(fp)
    pickle_dir = Path("alexandria_3d_pbe_pkls")
    pickle_dir.mkdir(exist_ok=True)
    for i, batch in tqdm(enumerate(batched(gener, limit))):
        with open(pickle_dir / f"{fp}__{i}.pkl", "wb") as f:
            dump(batch, f)

In [5]:
d.info

{'_name': ['alexandria_3d__file_alex_go_aaa_000__id_agm003286420__trajectory_0__frame_5'],
 'energy': -31.64143429,
 'forces': [[-0.00857753, -0.00743653, -0.0065639],
  [0.00857753, 0.00743653, 0.0065639],
  [-0.01433123, -0.01242486, 0.00961317],
  [0.00588325, -0.01532012, -0.01352241],
  [-0.00799872, -0.00693471, -0.00612097],
  [0.00799872, 0.00693471, 0.00612097],
  [-0.0, 0.0, -0.0],
  [-0.0, -0.0, -0.0],
  [-0.0, -0.0, -0.0],
  [0.00848522, -0.00978713, 0.0],
  [-0.00848522, 0.00121532, 0.00971138],
  [0.0, 0.00857182, -0.00971138],
  [-0.00848522, 0.00978713, -0.0],
  [0.00848522, -0.00121532, -0.00971138],
  [-0.0, -0.00857182, 0.00971138],
  [-0.00588325, 0.01532012, 0.01352241],
  [0.01433123, 0.01242486, -0.00961317],
  [0.01433123, -0.00799592, 0.01352241],
  [-0.01433123, 0.00799592, -0.01352241]],
 'stress': [[0.0185458, 0.07024299, 0.06200044],
  [0.07024299, -0.00157563, 0.053753],
  [0.06200044, 0.053753, -0.01502929]],
 'input': {'kpoints': [4, 4, 4],
  'ENAUG': 53

In [6]:
to_pickles(Path("alex_go_aaa_000.json"))

0it [00:00, ?it/s]

dict_keys(['agm003285455', 'agm003290242', 'agm003287487', 'agm003287244', 'agm003287135', 'agm003285864', 'agm003285861', 'agm003295313', 'agm003294941', 'agm003299737', 'agm003298484', 'agm003298637', 'agm003292885', 'agm003290257', 'agm003289503', 'agm003293994', 'agm003293579', 'agm003291170', 'agm003291689', 'agm003299742', 'agm003293142', 'agm003286061', 'agm003293739', 'agm003300111', 'agm003286100', 'agm003286330', 'agm003290558', 'agm003296029', 'agm003291676', 'agm003297943', 'agm003285810', 'agm003298167', 'agm003296110', 'agm003295990', 'agm003286972', 'agm003300139', 'agm003300110', 'agm003293218', 'agm003287486', 'agm003285827', 'agm003298201', 'agm003288872', 'agm003297300', 'agm003295268', 'agm003292435', 'agm003300614', 'agm003287177', 'agm003298923', 'agm003286962', 'agm003288254', 'agm003300147', 'agm003294208', 'agm003285463', 'agm003289695', 'agm003285527', 'agm003294098', 'agm003292057', 'agm003292886', 'agm003292436', 'agm003293221', 'agm003299374', 'agm003295956

27it [02:33,  5.68s/it]


In [3]:
with open("alexandria_3d_pbe_pkls/alex_go_aaa_000/alex_go_aaa_000__1.pkl", "rb") as f:
    for i in load(f):
        d = i
        break

In [29]:
with open(
    "/home/gpwolfe/colabfit/vast_ingest/alexandria/alex_go_aaa_000.json", "r"
) as f:
    data = json.load(f)

In [30]:
data.keys()

dict_keys(['agm003285455', 'agm003290242', 'agm003287487', 'agm003287244', 'agm003287135', 'agm003285864', 'agm003285861', 'agm003295313', 'agm003294941', 'agm003299737', 'agm003298484', 'agm003298637', 'agm003292885', 'agm003290257', 'agm003289503', 'agm003293994', 'agm003293579', 'agm003291170', 'agm003291689', 'agm003299742', 'agm003293142', 'agm003286061', 'agm003293739', 'agm003300111', 'agm003286100', 'agm003286330', 'agm003290558', 'agm003296029', 'agm003291676', 'agm003297943', 'agm003285810', 'agm003298167', 'agm003296110', 'agm003295990', 'agm003286972', 'agm003300139', 'agm003300110', 'agm003293218', 'agm003287486', 'agm003285827', 'agm003298201', 'agm003288872', 'agm003297300', 'agm003295268', 'agm003292435', 'agm003300614', 'agm003287177', 'agm003298923', 'agm003286962', 'agm003288254', 'agm003300147', 'agm003294208', 'agm003285463', 'agm003289695', 'agm003285527', 'agm003294098', 'agm003292057', 'agm003292886', 'agm003292436', 'agm003293221', 'agm003299374', 'agm003295956

In [31]:
len(data.keys())

10000

In [32]:
[len(data[id]) for id in data.keys()]

[3,
 3,
 3,
 3,
 3,
 3,
 3,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 2,
 3,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 2,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 2,
 3,
 3,
 3,
 3,
 2,
 3,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 2,
 1,
 3,
 3,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 2,
 3,
 3,
 2,
 3,
 3,
 3,
 3,
 2,
 3,
 3,
 3,
 3,
 3,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 2,
 3,
 2,
 3,
 3,
 3,
 3,
 2,
 3,
 3,
 3,
 3,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 2,
 3,
 3,
 2,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 2,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 2,
 3,
 3,
 3,
 2,
 2,
 3,
 2,
 3,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,


In [9]:
material_id = data["agm003272798"]

In [12]:
len(material_id)

3

In [33]:
material_id[0]

{'kpoints': [7, 2, 2],
 'PREC': 'high',
 'ENMAX': 264.4175,
 'ENAUG': 446.842,
 'steps': [{'structure': {'@module': 'pymatgen.core.structure',
    '@class': 'Structure',
    'charge': 0,
    'lattice': {'matrix': [[4.18620331, 0.0, 0.0],
      [0.0, 12.81988096, 0.0],
      [0.0, 0.0, 15.13173114]],
     'pbc': [True, True, True],
     'a': 4.18620331,
     'b': 12.81988096,
     'c': 15.13173114,
     'alpha': 90.0,
     'beta': 90.0,
     'gamma': 90.0,
     'volume': 812.0689877290661},
    'properties': {},
    'sites': [{'species': [{'element': 'Ba', 'occu': 1}],
      'abc': [0.75, 0.25968, 0.165083],
      'xyz': [3.1396524825, 3.3290666876928006, 2.49799157178462],
      'properties': {},
      'label': 'Ba'},
     {'species': [{'element': 'Ba', 'occu': 1}],
      'abc': [0.25, 0.24032, 0.665083],
      'xyz': [1.0465508275, 3.0808737923072003, 10.06385714178462],
      'properties': {},
      'label': 'Ba'},
     {'species': [{'element': 'Ba', 'occu': 1}],
      'abc': [0.25, 

In [34]:
material_id[0].keys()

dict_keys(['kpoints', 'PREC', 'ENMAX', 'ENAUG', 'steps'])

In [35]:
len(material_id[0]["steps"])

5

In [36]:
material_id[0]["steps"][0].keys()

dict_keys(['structure', 'energy', 'forces', 'stress'])

In [37]:
material_id[0]["steps"][0]

{'structure': {'@module': 'pymatgen.core.structure',
  '@class': 'Structure',
  'charge': 0,
  'lattice': {'matrix': [[4.18620331, 0.0, 0.0],
    [0.0, 12.81988096, 0.0],
    [0.0, 0.0, 15.13173114]],
   'pbc': [True, True, True],
   'a': 4.18620331,
   'b': 12.81988096,
   'c': 15.13173114,
   'alpha': 90.0,
   'beta': 90.0,
   'gamma': 90.0,
   'volume': 812.0689877290661},
  'properties': {},
  'sites': [{'species': [{'element': 'Ba', 'occu': 1}],
    'abc': [0.75, 0.25968, 0.165083],
    'xyz': [3.1396524825, 3.3290666876928006, 2.49799157178462],
    'properties': {},
    'label': 'Ba'},
   {'species': [{'element': 'Ba', 'occu': 1}],
    'abc': [0.25, 0.24032, 0.665083],
    'xyz': [1.0465508275, 3.0808737923072003, 10.06385714178462],
    'properties': {},
    'label': 'Ba'},
   {'species': [{'element': 'Ba', 'occu': 1}],
    'abc': [0.25, 0.74032, 0.834917],
    'xyz': [1.0465508275, 9.490814272307201, 12.633739568215379],
    'properties': {},
    'label': 'Ba'},
   {'species':

In [39]:
steps = material_id[0]["steps"]

In [40]:
[steps[i]["energy"] for i, step in enumerate(steps)]

[-155.21636357, -155.21646717, -155.21648417, -155.21650706, -155.21651992]

In [42]:
[steps[i]["forces"][0] for i, step in enumerate(steps)]

[[0.0, 0.00121322, -0.00013569],
 [0.0, 0.00102221, 0.00039485],
 [0.0, 0.0011993, 0.00046067],
 [0.0, 0.00092005, 0.00011045],
 [0.0, 0.00121448, 0.00090866]]

In [ ]:
cs_fps = sorted(list(dir_path_by_cs_id.rglob("configuration_set*")))
cs_fp = cs_fps[0]

print(cs_fp)
css = spark.read.parquet(str(cs_fp))
cs_names = [x[0] for x in css.select("cs_name").distinct().collect()]
co_ids = [x[0] for x in css.select("config_id").distinct().collect()]
print(len(co_ids))
with loader.session.transaction() as tx:
    table = tx.bucket("colabfit").schema("dev").table("co_wip")
    reader = table.select(predicate=table["id"].isin(co_ids), internal_row_id=True)
    print("batch")
    config_batch = reader.read_all()
    print(len(config_batch))
    configs = config_batch.to_struct_array().to_pandas()
configs = spark.createDataFrame(configs, schema=config_schema)

In [ ]:
string_cols = [
            "elements",
        ]
        unstring_col_udf = sf.udf(unstring_df_val, ArrayType(StringType()))
        for col in string_cols:
            config_df = config_df.withColumn(
                col, unstring_col_udf(sf.col(col))
            )
        unstring_col_udf = sf.udf(unstring_df_val, ArrayType(IntegerType()))
        int_cols = [
            "atomic_numbers",
            "dimension_types",
        ]
        for col in int_cols:
            config_df = config_df.withColumn(
                col, unstring_col_udf(sf.col(col))
            )
configs.cache()